# W01 — Research Question & Lane Framing

**FlyRank ML Internship · Applied Search Intelligence · Week 1 (Setup)**

The goal of this notebook is *not* a model — it is a defensible **question**. I frame the
chain **data → decision → action → cost of a wrong call**, pick a provisional lane, and back the
choice with a few real numbers from the starter dataset (`data/raw/content_refresh_anonymized.csv`).

> Lane choice is provisional and can be confirmed or changed until the end of Week 4. I am treating
> Week 1 as **research + test**, not a commitment.

## 1) My lane (and why)

**Provisional lane: Lane 3 — Structured Content Archetype Clustering.**

> *What distinct performance archetypes exist across the content inventory, and what action does
> each archetype suggest?*

**Why this lane (one paragraph).** The starter pipeline already treats every page with a single
1-D "refresh-first" score. But a single score collapses genuinely different situations into one
number: a high-traffic page that leaks clicks, a well-written page nobody can find, and a thin page
with no demand can all land near the same score for *opposite* reasons. Clustering asks a different,
more exploratory question — *are there natural groups of pages that behave alike?* — and maps each
group to its own playbook (protect / improve / rewrite / merge / prune / monitor). It is
**unsupervised**, so it fits Week 1 as research: I can inspect structure without committing to a
supervised label yet. This is **structured metric/metadata clustering, not semantic/text
clustering** — I use only safe observable signals and buckets, never raw URLs, titles, or queries.

## 2) The question: decision, action, cost of a wrong call

| Frame | This lane |
|---|---|
| **Search question** | What performance archetypes exist across the page inventory? |
| **Unit of analysis** | **One content item (page)**, described by safe 90-day observable signals + metadata. |
| **Output** | A small set of **named archetypes** (e.g. *champion*, *hidden gem*, *visible-but-leaking*, *engagement-problem*, *weak/no-demand*) with a profile and an **action mapping** per archetype. |
| **Action someone takes** | A content lead routes each archetype to the right **playbook** — protect / improve / rewrite / merge / prune / monitor — instead of triaging 30k pages one by one. |
| **Cost of a wrong recommendation** | Mistakes hit **in batches**: if an archetype is mislabeled, a whole group is routed to the wrong playbook — e.g. **pruning a hidden gem** (permanent traffic loss) or **spending rewrites on noise** (wasted capacity). Because clustering is unsupervised, clusters are *descriptive*, not ground truth — over-trusting them is the core risk. |

**Why data / ML can help at all.** With ~30k pages and several partly-independent signals (visibility,
position, CTR, engagement, freshness, length), no single human threshold captures the joint structure,
and eyeballing 30k rows is infeasible. Clustering surfaces candidate segments that a 1-D score can't —
*but* it needs validation (stability, silhouette, and human inspection of centroids) before any
archetype name or action is trusted. That "human judgment last" step is exactly why this is **not**
just "train a model."

## 3) Quick look at the data — 2–3 real numbers

I load the starter slice and show why Lane 3 looks worth the next 7 weeks: the inventory is
**heterogeneous** (number 1–2), the candidate signals are **not redundant** (number 2), and a quick
provisional clustering finds **separable, differentiated groups** (number 3).

All rate columns (`ctr`, `engagement_rate`, ...) are ×100 percentages; `avg_position == 0` means
*no position data*, not rank 0. Seeds are fixed for reproducibility.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path

# Robust path: walk up until we find the starter CSV, regardless of execution cwd.
root = Path.cwd()
while not (root / "data" / "raw" / "content_refresh_anonymized.csv").exists() and root != root.parent:
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA)
print(f"rows: {len(df):,}  |  columns: {df.shape[1]}  |  pseudonymized clients: {df['client_id'].nunique()}")

rows: 30,000  |  columns: 44  |  pseudonymized clients: 32


**Number 1 — the inventory spans the full performance range.** Pages are not one homogeneous pile:
visibility runs from `none` to `excellent`, and search position from `top_3` to `deep`. A single
"fix-this-first" score has to summarize all of that in one dimension — clustering keeps the dimensions.

In [2]:
imp = df["impression_tier"].value_counts(normalize=True).mul(100).round(1)
pos = df["position_tier"].value_counts(normalize=True).mul(100).round(1)
print("Visibility (impression_tier), % of pages:")
print(imp.to_string(), "\n")
print("Search position (position_tier), % of pages:")
print(pos.to_string())

Visibility (impression_tier), % of pages:
impression_tier
low          37.5
moderate     34.9
good         24.0
excellent     3.6 

Search position (position_tier), % of pages:
position_tier
page_1      39.4
striking    24.3
page_3_5    24.1
top_3        7.7
deep         4.4


**Number 2 — the candidate signals capture *different* things.** If visibility, position, CTR,
engagement, freshness, and length were all redundant, one score would suffice. The pairwise
correlations below are mostly weak, so pages vary along **several partly-independent axes** — the
structural precondition for meaningful archetypes.

In [3]:
feat = pd.DataFrame({
    "log_impr":          np.log1p(df["impressions_90d"]),
    "ctr":               df["ctr"],
    "avg_position":      df["avg_position"].replace(0, np.nan),   # 0 == "no position data"
    "engagement_rate":   df["engagement_rate"],
    "days_since_update": df["days_since_last_update"],
    "word_count":        df["word_count"],
})
print("Pairwise correlation of candidate signals:")
print(feat.corr(numeric_only=True).round(2).to_string())

Pairwise correlation of candidate signals:
                   log_impr   ctr  avg_position  engagement_rate  days_since_update  word_count
log_impr               1.00 -0.12         -0.07             0.07               0.21        0.34
ctr                   -0.12  1.00         -0.08             0.10              -0.02       -0.12
avg_position          -0.07 -0.08          1.00            -0.03               0.04        0.10
engagement_rate        0.07  0.10         -0.03             1.00              -0.01       -0.06
days_since_update      0.21 -0.02          0.04            -0.01               1.00        0.31
word_count             0.34 -0.12          0.10            -0.06               0.31        1.00


**Number 3 — a provisional clustering already finds separable, differentiated groups.** This is a
*preview*, not the capstone model: K-Means (k=5, `random_state=42`) on standardized safe features,
dropping rows with missing position/length. A positive silhouette means the groups are genuinely
separated, and the per-cluster medians show they are *qualitatively different* pages — the raw
material of archetypes.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cols = ["log_impr", "ctr", "avg_position", "engagement_rate", "days_since_update", "word_count"]
X = feat[cols].dropna()
Xs = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xs)
sil = silhouette_score(Xs, km.labels_)
print(f"pages clustered: {len(X):,}  (dropped rows missing position/word_count)")
print(f"k=5 silhouette score: {sil:.3f}   (>0 = real separation; k is NOT tuned yet)\n")

prof = X.copy()
prof["cluster"] = km.labels_
print("cluster sizes:")
print(prof["cluster"].value_counts().sort_index().to_string(), "\n")
print("per-cluster medians (archetype fingerprints, provisional):")
print(prof.groupby("cluster").median().round(2).to_string())

pages clustered: 21,109  (dropped rows missing position/word_count)
k=5 silhouette score: 0.278   (>0 = real separation; k is NOT tuned yet)

cluster sizes:
cluster
0     363
1    9228
2    3529
3    7853
4     136 

per-cluster medians (archetype fingerprints, provisional):
         log_impr    ctr  avg_position  engagement_rate  days_since_update  word_count
cluster                                                                               
0            5.76   0.17         10.60            44.44               20.0      2599.0
1            7.89   0.21          9.70             0.00               20.0      2847.0
2            8.20   0.13         19.00             0.29              104.0      5671.0
3            3.78   0.00         10.90             0.00               20.0      2608.0
4            1.39  33.33          3.65             0.00               20.0       893.0


**Reading the numbers.** The inventory is heterogeneous (Number 1), the signals are non-redundant
(Number 2), and even an untuned k=5 K-Means separates the pages into **differently-sized groups with
distinct fingerprints** (Number 3) — e.g. high-visibility/low-CTR pages sit apart from
low-visibility/well-engaged pages. That is enough evidence that *archetypes are there to be found*,
which is what makes Lane 3 worth the next 7 weeks. It is **not** proof that these five are the right
archetypes — k is untuned and the clusters are unnamed on purpose (see §4–§5).

## 4) Careful words: what I can and can't claim

**I can say (observed / directional / decision-support):**
- The starter inventory is heterogeneous across visibility, position, CTR, engagement, freshness, length.
- Candidate signals are weakly correlated, so pages vary along several axes.
- A provisional K-Means separates pages into distinct, differently-sized groups.

**I cannot (yet) claim:**
- That these are *the* archetypes — k is untuned, features unselected, clusters not stability-checked.
- That a cluster *is* a ground-truth type — clusters are **descriptive**, not labels.
- That an action *will* fix a page — no causal design; recommendations are *review/route*, not *guaranteed recovery*.
- Anything about Google's algorithm or AI rankings — out of scope for this data.

This is **structured metric/metadata clustering, not semantic clustering**. All outputs are
public-safe: pseudonymized IDs and aggregates only — no client names, domains, URLs, titles, or queries.

## 5) Self-check

- [x] **Picked a lane** — Lane 3 (Structured Content Archetype Clustering), declared provisional (changeable through Week 4).
- [x] **Named the decision and the action** — route each archetype to a playbook (protect / improve / rewrite / merge / prune / monitor).
- [x] **Named the cost of a wrong call** — batch mis-routing: pruning a hidden gem, or wasting rewrites on noise.
- [x] **Showed ≥ 2 real numbers** — inventory heterogeneity, signal non-redundancy, and a provisional k=5 silhouette + cluster fingerprints.
- [x] **Explained why this is not just "train a model"** — unsupervised structure discovery + routing that requires human validation (stability, inspection) before any name or action is trusted.
- [x] **Careful language** — observed / directional / decision-support; no causal or algorithm claims; public-safe.

**Leakage note for later weeks:** `trend_direction` / `trend_pct` are the supervised label source
and must never be clustering features; IDs are for grouping only. Product decision flags
(`health_score`, `priority_score`, `action_type`) are not in this data and must never be reintroduced
as features.

**Next steps (Weeks 2+):** feature selection + missingness handling per `content_type`, tune k
(elbow / silhouette), check cluster stability across seeds and client-holdout, profile and *then*
name archetypes, map each to an action, and scale to the warehouse release.